# Extract All Error Mappings

Loads every result file, classifies errors, and writes structured TSVs to `error_mappings/`.

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/data/jane/convert/math_gender/conversion_test')
OUT = PROJECT_ROOT / '3_error_taxonomy/error_mappings'
OUT.mkdir(exist_ok=True)

NON_REASONING = ['gpt-4o', 'qwen-coder', 'llama-4', 'gpt-oss-120b']
REASONING = ['gpt-5.2', 'deepseek-v3.1', 'qwen3-235b-thinking', 'claude-haiku-4-5']
ALL_MODELS = NON_REASONING + REASONING

CONDS = {
    'regular':   'results',
    'no_guide':  'results_no_guide',
    'math_only': 'results_math_only',
}

print(f"Models: {ALL_MODELS}")
print(f"Conditions: {list(CONDS.keys())}")

Models: ['gpt-4o', 'qwen-coder', 'llama-4', 'gpt-oss-120b', 'gpt-5.2', 'deepseek-v3.1', 'qwen3-235b-thinking', 'claude-haiku-4-5']
Conditions: ['regular', 'no_guide', 'math_only']


## 1. Load All Results

In [2]:
# load in desired file. pick full or only integers
df = pd.read_csv('/data/jane/convert/math_gender/conversion_test/2_analysis/integers.csv', sep='\t')

In [3]:
# # rename conditions 
# df['condition'] = df['condition'].replace({'math_only': 'math_only', 'no_guide': 'in_domain_no_guide', 'regular': 'in_domain_with_guide'})

## 2. Classify Errors

In [4]:
def classify_error(row):
    """Classify a wrong answer into an error type.

    Categories:
      correct          – loss == 0
      refusal          – no valid numeric answer AND refusal keywords in answer/response
      degenerate_loop  – model enters a repetitive loop (often ends in [TRUNCATED])
      changed_answer   – correct answer in reasoning trace but wrong answer in <answer> tags
      rounding         – loss <= 2 (small precision difference)
      decimal_shift    – answer differs by a power of 10
      magnitude_error  – answer is >100x or <0.01x the gold
      extra_factor     – off by a recognizable integer factor
      sign_error       – correct absolute value, wrong sign
      moderate_error   – loss 2..100 not fitting above
      large_error      – loss > 100 not fitting above
    """
    loss = row['loss']
    if pd.isna(loss) or loss == 0:
        return 'correct'

    # Refusal detection: no <answer> tags AND no [TRUNCATED] (truncated → degenerate_loop)
    _REFUSAL_KW = ['n/a', 'cannot', 'i cannot', 'unable to', 'not possible', 'impossible', 'i don\'t have realtime', 'don\'t have access', 'my knowledge was last updated', 'exchange rates fluctuate','real-time access','i apologize','have access']
    raw = str(row.get('raw_response', ''))
    if not re.search(r'<answer>', raw, re.IGNORECASE) and '[TRUNCATED]' not in raw:
        if any(kw in raw.lower() for kw in _REFUSAL_KW):
            return 'refusal'

    # Degenerate loop detection
    raw_text = str(row.get('raw_response', ''))
    has_truncated = '[TRUNCATED]' in raw_text
    clean_text = raw_text.replace('[TRUNCATED]', '').replace('[/REASONING]', '').replace('[REASONING]', '').strip()
    if has_truncated and len(clean_text) < 200:
        return 'degenerate_loop'
    if len(clean_text) >= 500:
        sample = clean_text[-3000:] if len(clean_text) > 3000 else clean_text
        _chunk = 50
        chunks = [sample[i:i+_chunk] for i in range(0, len(sample) - _chunk + 1, _chunk)]
        if len(chunks) >= 6:
            unique_ratio = len(set(chunks)) / len(chunks)
            if unique_ratio < (0.3 if has_truncated else 0.15):
                return 'degenerate_loop'

    # Changed answer: correct value in reasoning but wrong value in <answer> tags
    raw = str(row.get('raw_response', ''))
    answer_tags = re.findall(r'<answer>(.*?)</answer>', raw, re.IGNORECASE | re.DOTALL)
    if answer_tags:
        gold_str = str(row['answer'])
        try:
            gold_val = float(gold_str)
            last_tag_start = raw.rfind('<answer>')
            reasoning = raw[:last_tag_start] if last_tag_start > 0 else ''
            reasoning_nums = re.findall(r'-?\d+\.?\d*', reasoning)
            for n in reasoning_nums:
                try:
                    if gold_val != 0 and abs(float(n) - gold_val) / abs(gold_val) < 0.005:
                        return 'changed_answer'
                except ValueError:
                    continue
        except (ValueError, TypeError):
            pass

    if loss <= 2:
        return 'rounding'

    try:
        gold = float(row['answer'])
        ma = row.get('model_answer', np.nan)
        model_ans = float(ma) if pd.notna(ma) and str(ma) not in ('', 'nan', 'None') else np.nan
        if pd.isna(model_ans) or gold == 0:
            return 'large_error' if loss > 100 else 'moderate_error'

        ratio = model_ans / gold
        abs_ratio = abs(ratio)

        if abs(gold) > 0 and ((model_ans < 0 and gold > 0) or (model_ans > 0 and gold < 0)):
            if abs(abs(model_ans) - abs(gold)) / abs(gold) < 0.05:
                return 'sign_error'

        for power in range(-6, 7):
            if power == 0:
                continue
            if 0.9 < abs_ratio / (10 ** power) < 1.1:
                return 'decimal_shift'

        if abs_ratio > 100 or abs_ratio < 0.01:
            return 'magnitude_error'

        if model_ans != 0:
            g_over_m = gold / model_ans
            m_over_g = model_ans / gold
            if (abs(g_over_m - round(g_over_m)) < 0.05 and abs(round(g_over_m)) >= 2) or \
               (abs(m_over_g - round(m_over_g)) < 0.05 and abs(round(m_over_g)) >= 2):
                return 'extra_factor'
    except (ValueError, TypeError, ZeroDivisionError):
        pass

    return 'large_error' if loss > 100 else 'moderate_error'


df['error_type'] = df.apply(classify_error, axis=1)

print("Error type distribution:")
print(df['error_type'].value_counts().to_string())
print(f"\nCorrect: {(df['error_type']=='correct').sum():,} / {len(df):,} "
      f"({(df['error_type']=='correct').mean()*100:.1f}%)")

Error type distribution:
error_type
correct            1664723
moderate_error       89166
rounding             65786
decimal_shift        62135
changed_answer       33532
magnitude_error      30039
large_error          24991
extra_factor         13952
refusal               2368
degenerate_loop        191
sign_error               5

Correct: 1,664,723 / 1,986,888 (83.8%)


In [5]:
df[df['error_type'] == 'refusal'].sample(100).to_csv('error_mappings/examples_refusal.csv', sep='\t', index=False)

## 3. Prepare Shared Filters

In [6]:
# No-distractor subset (used by several extractions)
nodist_mask = (
    df['distractor'].astype(str).isin(['null', 'nan', '', 'None'])
    | df['distractor'].isna()
)
df_nd = df[nodist_mask].copy()

# Condition subsets
mo = df[df['condition'] == 'math_only'].copy()
ng = df[df['condition'] == 'in_domain_no_guide'].copy()
reg = df[df['condition'] == 'in_domain_with_guide'].copy()

# No-distractor + no-clothing for math/no-guide comparison
ng_clean = ng[~ng['domain'].str.contains('clothing')].copy()
for d in [mo, ng_clean]:
    mask = d['distractor'].astype(str).isin(['null', 'nan', '', 'None']) | d['distractor'].isna()
    d.drop(d[~mask].index, inplace=True)

mo['key'] = mo['model'] + '|' + mo['domain'] + '|' + mo['number'].astype(str) + '|' + mo['answer'].astype(str)
ng_clean['key'] = ng_clean['model'] + '|' + ng_clean['domain'] + '|' + ng_clean['number'].astype(str) + '|' + ng_clean['answer'].astype(str)

# No-distractor subsets of reg/ng for guide comparison
reg_nd = df_nd[df_nd['condition'] == 'in_domain_with_guide'].copy()
ng_nd = df_nd[df_nd['condition'] == 'in_domain_no_guide'].copy()
reg_nd['key'] = reg_nd['model'] + '|' + reg_nd['domain'] + '|' + reg_nd['number'].astype(str) + '|' + reg_nd['answer'].astype(str)
ng_nd['key'] = ng_nd['model'] + '|' + ng_nd['domain'] + '|' + ng_nd['number'].astype(str) + '|' + ng_nd['answer'].astype(str)

print(f"No-distractor rows: {len(df_nd):,}")
print(f"Math-only (no dist): {len(mo):,}")
print(f"No-guide (no dist, no clothing): {len(ng_clean):,}")

No-distractor rows: 546,888
Math-only (no dist): 180,840
No-guide (no dist, no clothing): 180,840


## 4. Extract: Error Rate Summary

One row per model × domain × condition with accuracy, error rate, and loss statistics.

In [7]:
summary = df.groupby(['model', 'domain', 'condition']).agg(
    total=('loss', 'size'),
    n_correct=('loss', lambda x: (x == 0).sum()),
    n_wrong=('loss', lambda x: (x > 0).sum()),
    mean_loss=('loss', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
    median_loss=('loss', lambda x: x[x > 0].median() if (x > 0).any() else 0),
    max_loss=('loss', 'max'),
).reset_index()
summary['accuracy_pct'] = (summary['n_correct'] / summary['total'] * 100).round(2)
summary['error_pct'] = (summary['n_wrong'] / summary['total'] * 100).round(2)
summary.to_csv(OUT / 'error_rate_summary.tsv', sep='\t', index=False)
print(f"error_rate_summary.tsv: {len(summary)} rows")
summary.head()

error_rate_summary.tsv: 256 rows


,model,domain,condition,total,n_correct,n_wrong,mean_loss,median_loss,max_loss,accuracy_pct,error_pct
0,claude-haiku-4-5,bits_bytes,in_domain_no_guide,600,468,132,3.300805,2.357143,6.867743,78.00,22.00
1,claude-haiku-4-5,bits_bytes,in_domain_with_guide,600,598,2,99.450000,99.450000,99.900000,99.67,0.33
2,claude-haiku-4-5,bits_bytes,math_only,600,600,0,0.000000,0.000000,0.000000,100.00,0.00
3,claude-haiku-4-5,clothing_size,in_domain_no_guide,273,54,216,1.000000,1.000000,1.000000,19.78,79.12
4,claude-haiku-4-5,clothing_size,in_domain_with_guide,273,272,1,1.000000,1.000000,1.000000,99.63,0.37


## 5. Extract: Error Type Breakdown

Count of each error type per model × domain × condition.

In [8]:
etype_counts = (
    df[df['error_type'] != 'correct']
    .groupby(['model', 'domain', 'condition', 'error_type'])
    .size()
    .reset_index(name='count')
)
etype_counts.to_csv(OUT / 'error_type_breakdown.tsv', sep='\t', index=False)
print(f"error_type_breakdown.tsv: {len(etype_counts)} rows")
etype_counts.head()

error_type_breakdown.tsv: 833 rows


,model,domain,condition,error_type,count
0,claude-haiku-4-5,bits_bytes,in_domain_no_guide,changed_answer,16
1,claude-haiku-4-5,bits_bytes,in_domain_no_guide,moderate_error,115
2,claude-haiku-4-5,bits_bytes,in_domain_no_guide,rounding,1
3,claude-haiku-4-5,bits_bytes,in_domain_with_guide,decimal_shift,2
4,claude-haiku-4-5,clothing_size,in_domain_no_guide,changed_answer,75


## 6. Extract: Math-Only Correct → No-Guide Wrong

Every case where the model solved the pure arithmetic but failed the domain-phrased version.

In [9]:
mo_correct_keys = set(mo[mo['loss'] == 0]['key'].values)
ng_wrong = ng_clean[ng_clean['loss'] > 0].copy()
ng_wrong['math_was_correct'] = ng_wrong['key'].isin(mo_correct_keys)
disc = ng_wrong[ng_wrong['math_was_correct']].copy()

disc_out = disc[['model', 'domain', 'number', 'answer', 'loss', 'error_type',
                 'prompt', 'raw_response']].copy()
disc_out['raw_response'] = disc_out['raw_response'].str[:500]
disc_out['prompt'] = disc_out['prompt'].str[:300]
disc_out.to_csv(OUT / 'math_correct_noguide_wrong.tsv', sep='\t', index=False)
print(f"math_correct_noguide_wrong.tsv: {len(disc_out):,} rows")

# Summary per model × domain
disc_summary = disc.groupby(['model', 'domain']).agg(
    n_discrepant=('loss', 'size'),
    mean_loss=('loss', 'mean'),
    median_loss=('loss', 'median'),
).reset_index()
ng_totals = ng_clean.groupby(['model', 'domain']).size().reset_index(name='ng_total')
disc_summary = disc_summary.merge(ng_totals, on=['model', 'domain'], how='left')
disc_summary['disc_pct'] = (disc_summary['n_discrepant'] / disc_summary['ng_total'] * 100).round(2)
disc_summary.to_csv(OUT / 'math_vs_noguide_summary.tsv', sep='\t', index=False)
print(f"math_vs_noguide_summary.tsv: {len(disc_summary)} rows")
disc_summary

math_correct_noguide_wrong.tsv: 36,020 rows
math_vs_noguide_summary.tsv: 54 rows


,model,domain,n_discrepant,mean_loss,median_loss,ng_total,disc_pct
0,claude-haiku-4-5,bits_bytes,132,3.300805e+00,2.357143,600,22.00
1,claude-haiku-4-5,cooking,59,5.755213e-01,0.295057,600,9.83
2,claude-haiku-4-5,currency,11007,7.414711e+04,12.472222,11200,98.28
3,claude-haiku-4-5,density,38,2.655385e+03,0.115261,600,6.33
4,claude-haiku-4-5,energy,41,1.464160e+04,0.136575,1000,4.10
5,claude-haiku-4-5,moles_to_particles,193,1.000000e+02,100.000000,200,96.50
6,claude-haiku-4-5,speed,12,1.748792e-01,0.163535,600,2.00
7,claude-haiku-4-5,temperature,2,1.817989e-01,0.181799,600,0.33
8,claude-haiku-4-5,timezone,1141,2.098160e+02,120.000000,4320,26.41
9,claude-haiku-4-5,volume,27,3.963212e-01,0.159934,600,4.50


## 7. Extract: Cross-Domain Errors

Same input number correct in domain A, wrong in domain B (no-distractor only).

In [10]:
cross_rows = []
for model in df_nd['model'].unique():
    for cond in ['in_domain_with_guide', 'in_domain_no_guide', 'math_only']:
        sub = df_nd[(df_nd['model'] == model) & (df_nd['condition'] == cond)]
        if len(sub) == 0:
            continue
        correct = sub[sub['loss'] == 0]
        wrong = sub[sub['loss'] > 2]

        correct_nums = correct.groupby('number')['domain'].apply(set).to_dict()

        for _, w in wrong.iterrows():
            num = w['number']
            if num in correct_nums:
                c_doms = correct_nums[num] - {w['domain']}
                if c_doms:
                    cross_rows.append({
                        'model': model,
                        'condition': cond,
                        'number': num,
                        'wrong_domain': w['domain'],
                        'wrong_gold': w['answer'],
                        'wrong_loss': w['loss'],
                        'wrong_error_type': w['error_type'],
                        'correct_domains': ','.join(sorted(c_doms)[:5]),
                        'n_correct_domains': len(c_doms),
                        'wrong_response_snippet': str(w['raw_response'])[:200],
                        'wrong_prompt_snippet': str(w['prompt'])[:200],
                    })

cross_df = pd.DataFrame(cross_rows)
cross_df.to_csv(OUT / 'cross_domain_errors.tsv', sep='\t', index=False)
print(f"cross_domain_errors.tsv: {len(cross_df):,} rows")

cross_summary = cross_df.groupby(['model', 'condition', 'wrong_domain']).agg(
    n_cross_errors=('wrong_loss', 'size'),
    mean_loss=('wrong_loss', 'mean'),
).reset_index()
# cross_summary.to_csv(OUT / 'cross_domain_summary.tsv', sep='\t', index=False)
print(f"cross_domain_summary.tsv: {len(cross_summary)} rows")

cross_domain_errors.tsv: 81,329 rows
cross_domain_summary.tsv: 149 rows


## 8. Extract: All Individual Errors

In [11]:
# err_cols = ['model', 'domain', 'condition', 'number', 'answer', 'loss', 'error_type', 'distractor']
# if 'model_answer' in df.columns:
#     err_cols.append('model_answer')

all_errors = df[df['error_type'] != 'correct'].copy()
all_errors.to_csv(OUT / 'all_errors.tsv', sep='\t', index=False)
print(f"all_errors columns: {all_errors.columns.tolist()}")
print(f"all_errors.tsv: {len(all_errors):,} rows")

all_errors columns: ['domain', 'distractor', 'prompt', 'number', 'answer', 'difficulty', 'raw_response', 'model_answer', 'loss', 'reasoning_tokens', 'call_seconds', 'model', 'condition', 'frequency', 'frequency_level', 'is_reasoning', 'display_name', 'is_correct', 'num_decimals', 'num_input_decimals', 'frequency_difficulty', 'error_type']
all_errors.tsv: 322,165 rows


In [12]:
# sample 30 per domain and save

all_errors.groupby('domain').sample(30).to_csv(OUT / 'examples_to_analyze.csv', sep='\t', index=False)

In [13]:
all_errors[all_errors['error_type'] == 'refusal'].sample(30).to_csv(OUT / 'examples_refusal.csv', sep='\t', index=False)

In [14]:
all_errors[all_errors['is_reasoning'] == True].groupby('domain').sample(30).to_csv(OUT / 'examples_reasoning.csv', sep='\t', index=False)

In [15]:
all_errors[all_errors['error_type'] == 'extra_factor'].sample(30)

,domain,distractor,prompt,number,answer,difficulty,raw_response,model_answer,loss,reasoning_tokens,...,condition,frequency,frequency_level,is_reasoning,display_name,is_correct,num_decimals,num_input_decimals,frequency_difficulty,error_type
1752586,volume,spit,Convert 250 quarts of spit to tablespoons. Pro...,250,15999.972792398908,Easy,<answer>4000</answer>,4000.0,74.999957,NaN,...,in_domain_no_guide,5358253.0,common,False,Llama-4-Maverick,False,0,0,hard,extra_factor
1634863,volume,bubble gum,Convert 75 quarts of bubble gum to tablespoons...,75,4799.991837719672,Easy,<answer>1200</answer>,1200.0,74.999957,0.0,...,in_domain_no_guide,10590900.0,common,False,NaN,False,0,0,hard,extra_factor
1953800,moles_to_particles,bubble gum,Convert 20 moles of bubble gum to atoms. Provi...,20,1.204428152e+25,Easy,"[REASONING]\nOkay, the user wants me to conver...",5.42e+26,4400.060872,NaN,...,in_domain_no_guide,71043598.0,very_common,True,Qwen3-235B-Think ★,False,6,0,hard,extra_factor
870415,moles_to_particles,glucose,Convert 55 moles of glucose to atoms.\n\nConve...,55.0,3.312177418e+25,Easy,"[REASONING]\nOkay, let's see. I need to conver...",7.9492258032e+26,2300.000000,NaN,...,in_domain_with_guide,10725928.0,common,True,Qwen3-235B-Think ★,False,14,0,hard,extra_factor
536255,moles_to_particles,glucose,Convert 90 moles of glucose to atoms.\n\nConve...,90,5.419926684e+25,Easy,<answer>1.3008e27</answer>,1.3008e+27,2300.032465,0.0,...,in_domain_with_guide,15685922.0,very_common,False,NaN,False,8,0,hard,extra_factor
1224469,density,coffee,Convert 11 lb/ft³s of coffee to g/cm³s. Provid...,11.0,0.176203,Easy,<answer>0.0088</answer>,0.0088,95.005763,0.0,...,in_domain_no_guide,76986440.0,very_common,True,DeepSeek-V3.1 ★,False,4,0,hard,extra_factor
1768473,cooking,flour,Convert 99 cups of flour to teaspoons. Provide...,99,4752.0,Easy,<answer>1584.0</answer>,1584.0,66.666667,0.0,...,in_domain_no_guide,7752192.0,common,False,Qwen3-Coder-480B,False,0,0,hard,extra_factor
1464845,density,RedBull,Convert 12 lb/in³s of RedBull to kg/m³s. Provi...,12.0,332158.856523,Easy,<answer>5317159.363927999</answer>,5317159.363928,1500.788075,NaN,...,in_domain_no_guide,89560510.0,very_common,True,GPT-5.2 ★,False,9,0,hard,extra_factor
1387115,moles_to_particles,water,Convert 900 moles of water to atoms. Provide o...,900,5.419926684e+26,Easy,<answer>1.0848e+27</answer>,1.0848e+27,100.150309,NaN,...,in_domain_no_guide,1757767.0,common,False,GPT-4o,False,8,0,hard,extra_factor
1406847,volume,cubane,Convert 40 quarts of cubane to tablespoons. Pr...,40,2559.995646783825,Easy,<answer>30720</answer>,30720.0,1100.002041,NaN,...,in_domain_no_guide,28602679.0,very_common,False,GPT-4o,False,0,0,hard,extra_factor


In [16]:
all_errors[all_errors['error_type'] == 'refusal'].sample(30)

,domain,distractor,prompt,number,answer,difficulty,raw_response,model_answer,loss,reasoning_tokens,...,condition,frequency,frequency_level,is_reasoning,display_name,is_correct,num_decimals,num_input_decimals,frequency_difficulty,error_type
1169654,moles_to_particles,matcha latte,Convert 700 moles of matcha latte to atoms. Pr...,700,4.215498532e+26,Easy,[REASONING]\nThis is an interesting question b...,10.0,1.000000e+02,495.0,...,in_domain_no_guide,2415427.0,common,False,NaN,False,0,0,hard,refusal
1113631,currency,NaN,Convert 9000 UZS to SLL. Provide only the nume...,9000,17.985306122448982,Easy,[REASONING]\nI need to convert 9000 UZS (Uzbek...,9000.0,4.994085e+04,581.0,...,in_domain_no_guide,510501.0,uncommon,False,NaN,False,0,0,hard,refusal
1175817,timezone,NaN,Convert 9:00 AM in Zhemuzhen time to Yorba Lin...,9:00 AM,5:00 PM,Easy,[REASONING]\nThis is a tricky question. Let me...,4PM,6.000000e+01,254.0,...,in_domain_no_guide,0.0,not_found,False,NaN,False,0,0,hard,refusal
1107755,currency,NaN,Convert 19 CHF to GBP. Provide only the numeri...,19,17.78205128205128,Easy,[REASONING]\nI need to convert 19 CHF (Swiss F...,14.06,2.093151e+01,295.0,...,in_domain_no_guide,37776077.0,very_common,False,NaN,False,2,0,hard,refusal
1105285,currency,NaN,Convert 100 GBP to CHF. Provide only the numer...,100,106.84931506849315,Easy,[REASONING]\nThe user is asking me to convert ...,-123.0,2.151154e+02,514.0,...,in_domain_no_guide,47011252.0,very_common,False,NaN,False,0,0,hard,refusal
1110024,currency,NaN,Convert 2500 IRR to SLL. Provide only the nume...,2500,0.0514410210202132,Easy,[REASONING]\nI need to convert 2500 IRR (Irani...,1323.0,2.571777e+06,509.0,...,in_domain_no_guide,1111798.0,uncommon,False,NaN,False,0,0,hard,refusal
1114458,currency,NaN,Convert 22 GNF to UZS. Provide only the numeri...,22,30.79648,Easy,[REASONING]\nI need to convert 22 GNF (Guinean...,-31.0,2.006609e+02,549.0,...,in_domain_no_guide,31485618.0,very_common,False,NaN,False,0,0,hard,refusal
1152567,energy,yogurt,Convert 30 kilowatt-hours of yogurt to BTUs. P...,30,102364.5265096109,Easy,[REASONING]\nThis question is asking me to con...,3.412,9.999667e+01,458.0,...,in_domain_no_guide,52407860.0,very_common,False,NaN,False,3,0,hard,refusal
1110043,currency,NaN,Convert 7 IRR to IDR. Provide only the numeric...,7,0.0986474037414096,Easy,[REASONING]\nThe user is asking me to convert ...,7.0,6.995980e+03,551.0,...,in_domain_no_guide,123335391.0,very_common,False,NaN,False,0,0,easy,refusal
1109467,currency,NaN,Convert 40 LBP to UZS. Provide only the numeri...,40,5.727644652250146,Easy,[REASONING]\nI need to convert 40 Lebanese Pou...,40.0,5.983673e+02,517.0,...,in_domain_no_guide,28602679.0,very_common,False,NaN,False,0,0,hard,refusal


## 9. Extract: Guide vs No-Guide Comparison

- **guide_helps**: correct with guide, wrong without  
- **guide_hurts**: wrong with guide, correct without

In [17]:
# Guide helps: with-guide correct, no-guide wrong
reg_correct_keys = set(reg_nd[reg_nd['loss'] == 0]['key'].values)
ng_nd_wrong = ng_nd[ng_nd['loss'] > 0].copy()
ng_nd_wrong['guide_was_correct'] = ng_nd_wrong['key'].isin(reg_correct_keys)
guide_helps = ng_nd_wrong[ng_nd_wrong['guide_was_correct']]

guide_summary = guide_helps.groupby(['model', 'domain']).agg(
    n_guide_helps=('loss', 'size'),
    mean_loss=('loss', 'mean'),
).reset_index()
ng_nd_totals = ng_nd.groupby(['model', 'domain']).size().reset_index(name='ng_total')
guide_summary = guide_summary.merge(ng_nd_totals, on=['model', 'domain'], how='left')
guide_summary['guide_helps_pct'] = (guide_summary['n_guide_helps'] / guide_summary['ng_total'] * 100).round(2)
guide_summary.to_csv(OUT / 'guide_vs_noguide_summary.tsv', sep='\t', index=False)
print(f"guide_vs_noguide_summary.tsv: {len(guide_summary)} rows")

# Guide hurts: with-guide wrong, no-guide correct
ng_correct_keys = set(ng_nd[ng_nd['loss'] == 0]['key'].values)
reg_nd_wrong = reg_nd[reg_nd['loss'] > 0].copy()
reg_nd_wrong['noguide_was_correct'] = reg_nd_wrong['key'].isin(ng_correct_keys)
guide_hurts = reg_nd_wrong[reg_nd_wrong['noguide_was_correct']]

guide_hurts_summary = guide_hurts.groupby(['model', 'domain']).size().reset_index(name='n_guide_hurts')
# guide_hurts_summary.to_csv(OUT / 'guide_hurts_cases.tsv', sep='\t', index=False)
# print(f"guide_hurts_cases.tsv: {len(guide_hurts_summary)} rows")

display(guide_summary)
display(guide_hurts_summary)

guide_vs_noguide_summary.tsv: 80 rows


,model,domain,n_guide_helps,mean_loss,ng_total,guide_helps_pct
0,claude-haiku-4-5,bits_bytes,130,3.280316,600,21.67
1,claude-haiku-4-5,clothing_size,215,1.000000,273,78.75
2,claude-haiku-4-5,cooking,27,1.024359,600,4.50
3,claude-haiku-4-5,currency,9434,70505.439883,11200,84.23
4,claude-haiku-4-5,density,22,4586.479363,600,3.67
...,...,...,...,...,...,...
75,qwen3-235b-thinking,density,34,0.367149,600,5.67
76,qwen3-235b-thinking,energy,55,0.686261,1000,5.50
77,qwen3-235b-thinking,speed,2,1750.057042,600,0.33
78,qwen3-235b-thinking,temperature,2,0.290909,600,0.33


,model,domain,n_guide_hurts
0,claude-haiku-4-5,cooking,1
1,claude-haiku-4-5,currency,4
2,claude-haiku-4-5,density,80
3,claude-haiku-4-5,energy,19
4,claude-haiku-4-5,speed,44
...,...,...,...
56,qwen3-235b-thinking,density,13
57,qwen3-235b-thinking,energy,36
58,qwen3-235b-thinking,speed,2
59,qwen3-235b-thinking,temperature,7


## 10. Extract: Distractor Impact

In [18]:
df['has_distractor'] = ~(
    df['distractor'].astype(str).isin(['null', 'nan', '', 'None'])
    | df['distractor'].isna()
)
domains_with_dist = sorted(df[df['has_distractor']]['domain'].unique())
print(f"Domains with distractors: {domains_with_dist}")

dist_rows = []
for dom in domains_with_dist:
    for m in df['model'].unique():
        for cond in ['regular', 'no_guide']:
            for has_d in [True, False]:
                sub = df[(df['domain'] == dom) & (df['model'] == m)
                         & (df['condition'] == cond) & (df['has_distractor'] == has_d)]
                if len(sub) == 0:
                    continue
                dist_rows.append({
                    'domain': dom, 'model': m, 'condition': cond,
                    'has_distractor': has_d,
                    'n': len(sub),
                    'n_correct': int((sub['loss'] == 0).sum()),
                    'n_wrong': int((sub['loss'] > 0).sum()),
                    'error_pct': round((sub['loss'] > 0).mean() * 100, 2),
                    'mean_loss_when_wrong': round(sub[sub['loss'] > 0]['loss'].mean(), 2)
                                           if (sub['loss'] > 0).any() else 0,
                })

dist_df = pd.DataFrame(dist_rows)
# dist_df.to_csv(OUT / 'distractor_impact.tsv', sep='\t', index=False)
# print(f"distractor_impact.tsv: {len(dist_df)} rows")
dist_df.head(10)

Domains with distractors: ['cooking', 'density', 'energy', 'moles_to_particles', 'volume']


""


## 11. Summary

Print all output files and key numbers.

In [19]:
print(f"Output directory: {OUT.resolve()}")
print(f"{'File':<40s} {'Rows':>10s}  {'Size':>8s}")
print('─' * 62)
for f in sorted(OUT.glob('*.tsv')):
    nrows = sum(1 for _ in open(f)) - 1
    size_kb = f.stat().st_size / 1024
    print(f"{f.name:<40s} {nrows:>10,d}  {size_kb:>7.0f} KB")

print(f"\n--- Quick stats ---")
print(f"Total rows loaded:       {len(df):>12,d}")
print(f"Total errors:            {(df['error_type']!='correct').sum():>12,d}")
print(f"Math-correct/NG-wrong:   {len(disc_out):>12,d}")
print(f"Cross-domain errors:     {len(cross_df):>12,d}")
print(f"Guide-helps cases:       {guide_summary['n_guide_helps'].sum():>12,.0f}")
print(f"Guide-hurts cases:       {guide_hurts_summary['n_guide_hurts'].sum():>12,d}")

Output directory: /data/jane/convert/math_gender/conversion_test/3_error_taxonomy/error_mappings
File                                           Rows      Size
──────────────────────────────────────────────────────────────
all_errors.tsv                            7,534,366   372788 KB
cross_domain_errors.tsv                     319,862    22068 KB
cross_domain_summary.tsv                        181        9 KB
distractor_impact.tsv                           134        8 KB
error_rate_summary.tsv                          256       27 KB
error_type_breakdown.tsv                        833       46 KB
examples.tsv                                  6,709      359 KB
examples_to_analyze.tsv                      11,514      550 KB
guide_hurts_cases.tsv                            52        1 KB
guide_vs_noguide_summary.tsv                     80        4 KB
math_correct_noguide_wrong.tsv              208,088    15331 KB
math_vs_noguide_summary.tsv                      54        4 KB

--- Quick